# 06 - Model Training and Evaluation

This notebook trains the first leakage-reviewed default prediction models for the Canadian retail credit risk project.

The goal is to compare a transparent baseline model against stronger ensemble candidates while keeping the modelling process auditable, reproducible, and aligned with credit-risk model governance expectations.

## Professional framing

This notebook uses the modelling dataset created in Notebook 05. All imputation, scaling, and one-hot encoding are fit inside scikit-learn pipelines using the training split only.

Model comparison focuses on credit-risk relevant metrics:

1. **PR-AUC** because defaults are the minority class.
2. **ROC-AUC** for general discriminatory power.
3. **Recall** because missed defaults can be costly.
4. **Precision and review rate** because too many false alerts can overload credit-review teams.
5. **Brier score** because calibrated probabilities matter for risk ranking and portfolio monitoring.
6. **MCC and balanced accuracy** because accuracy alone is misleading when the default rate is low.

Threshold optimization is deferred to Notebook 07. Here, models are evaluated at the default 0.50 threshold and ranked primarily by validation PR-AUC.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from credit_risk.config import MODEL_ARTIFACT_DIR, PROCESSED_DIR, TABLE_DIR
from credit_risk.models.train import (
    infer_feature_columns,
    save_model_training_artifacts,
    train_and_evaluate_candidate_models,
)

MODELING_PATH = PROCESSED_DIR / "credit_risk_modeling_dataset.csv"
MODELING_PATH

## Load leakage-reviewed modelling dataset

In [ ]:
modeling_df = pd.read_csv(MODELING_PATH, low_memory=False)

load_summary = {
    "row_count": modeling_df.shape[0],
    "column_count": modeling_df.shape[1],
    "target_default_rate": float(modeling_df["defaulter"].mean()),
    "record_key_duplicate_count": int(modeling_df.duplicated(["user_id", "record_sequence"]).sum()),
    "split_values": sorted(modeling_df["split"].dropna().unique().tolist()),
}
load_summary

The row count, target rate, and split labels should match Notebook 05. Stop before training if this check does not pass.

In [ ]:
split_check = (
    modeling_df.groupby("split")
    .agg(
        row_count=("defaulter", "size"),
        default_count=("defaulter", "sum"),
        default_rate=("defaulter", "mean"),
    )
    .reset_index()
)
split_check["default_rate_pct"] = split_check["default_rate"] * 100
split_check

## Feature set used for model training

In [ ]:
numeric_features, categorical_features = infer_feature_columns(modeling_df)

feature_summary = {
    "numeric_features": len(numeric_features),
    "categorical_features": len(categorical_features),
    "total_features": len(numeric_features) + len(categorical_features),
}
feature_summary

In [ ]:
feature_catalog = pd.DataFrame(
    {
        "feature": numeric_features + categorical_features,
        "feature_type": ["numeric"] * len(numeric_features) + ["categorical"] * len(categorical_features),
    }
)
feature_catalog.head(20)

## Train candidate models

Candidate models:

1. **Class-weighted Logistic Regression** as the interpretable benchmark.
2. **Class-weighted Random Forest** as a nonlinear ensemble challenger.
3. **Weighted XGBoost** as the primary gradient-boosting challenger.

The models are fitted only on the training split. Validation and test splits are not used to fit preprocessing steps.

In [ ]:
artifacts = train_and_evaluate_candidate_models(modeling_df, random_state=42)

print(f"Trained models: {list(artifacts.fitted_models.keys())}")
print(f"Champion by validation ranking: {artifacts.champion_model_name}")

## Validation performance at default 0.50 threshold

Validation results are used for model selection. PR-AUC is prioritized because default is a minority class and the business cares about ranking risky borrowers for review.

In [ ]:
validation_results = artifacts.validation_results.copy()
validation_results

In [ ]:
artifacts.selection_summary

## Test-set performance

The test split is kept as a final holdout view. The selected champion should not be changed based on the test table unless there is evidence of severe instability.

In [ ]:
test_results = artifacts.test_results.copy()
test_results

## Compare ranking metrics

In [ ]:
plot_df = validation_results.sort_values("pr_auc", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(plot_df["model_name"], plot_df["pr_auc"])
ax.set_title("Validation PR-AUC by Candidate Model")
ax.set_xlabel("PR-AUC")
ax.set_ylabel("Model")
for idx, value in enumerate(plot_df["pr_auc"]):
    ax.text(value, idx, f" {value:.4f}", va="center")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = validation_results.sort_values("roc_auc", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(plot_df["model_name"], plot_df["roc_auc"])
ax.set_title("Validation ROC-AUC by Candidate Model")
ax.set_xlabel("ROC-AUC")
ax.set_ylabel("Model")
for idx, value in enumerate(plot_df["roc_auc"]):
    ax.text(value, idx, f" {value:.4f}", va="center")
plt.tight_layout()
plt.show()

## Confusion-matrix review at 0.50 threshold

This table translates model performance into operational risk. A model with strong ranking metrics can still need threshold tuning if the default 0.50 cutoff produces too few or too many review cases.

In [ ]:
confusion_cols = [
    "model_name",
    "dataset",
    "threshold",
    "true_positive",
    "false_positive",
    "false_negative",
    "true_negative",
    "recall",
    "precision",
    "review_rate",
    "business_cost",
]
validation_results[confusion_cols]

## Save model artifacts and experiment tables

The saved outputs support later threshold selection, explainability, and governance documentation.

In [ ]:
save_model_training_artifacts(artifacts, TABLE_DIR, MODEL_ARTIFACT_DIR)

saved_outputs = {
    "validation_results": str(TABLE_DIR / "model_validation_results_default_threshold.csv"),
    "test_results": str(TABLE_DIR / "model_test_results_default_threshold.csv"),
    "selection_summary": str(TABLE_DIR / "model_selection_summary.csv"),
    "validation_predictions": str(TABLE_DIR / "validation_predictions_default_threshold.csv"),
    "test_predictions": str(TABLE_DIR / "test_predictions_default_threshold.csv"),
    "candidate_models": str(MODEL_ARTIFACT_DIR / "candidate_models.joblib"),
    "champion_model": str(MODEL_ARTIFACT_DIR / "champion_model.joblib"),
    "feature_metadata": str(MODEL_ARTIFACT_DIR / "model_feature_metadata.joblib"),
}
saved_outputs

## Notebook 06 conclusions

Carry these decisions forward into Notebook 07:

1. The champion model is selected using validation PR-AUC, with ROC-AUC, calibration, MCC, recall, precision, and review rate as supporting evidence.
2. The default 0.50 threshold is not expected to be the final business threshold for a credit-risk review strategy.
3. Notebook 07 should choose a business threshold using recall, precision, review-capacity, and estimated cost trade-offs.
4. Notebook 08 should explain the selected champion model using SHAP, local explanations, and business-friendly reason codes.
5. Notebook 09 should convert model results into governance artifacts: model card, validation summary, limitations, monitoring plan, and stakeholder brief.